# Residual × Depth Ablation Study (BN fixed ON)

**Research question (scoped-down version):** With BatchNorm held on (the realistic regime —
almost nobody trains CNNs without it in practice), does the marginal contribution of residual
connections to CIFAR-10 test accuracy change between a shallow (depth=8) and a moderately
deeper (depth=20) network?

**Scope note:** This is a deliberately small design (2 depths × Residual on/off × 3 seeds = 12 runs
per training recipe), chosen to fit a tight compute/time budget. It **drops** the BN on/off comparison
(H1a/H1b) entirely — that axis is left as future work. What this design **does** test directly is a gap
found during the related-works review: whether residual's marginal benefit is depth-dependent even
when BN is present. Prior CIFAR-10 skip-connection ablations start at 18-20 layers (He et al. 2016;
Liu & Goh 2025, arXiv:2510.24036), while the original HW2 Section 4 comparison (~10 layers, two
different architectures, one run each) is uninterpretable as a statement about skip connections.

**Two training recipes (`SCHEDULE` switch in the experiment cell).**

- `constant` (**v1, already run**): Adam, lr = 1e-3 held constant, 10 epochs. The legacy "test accuracy"
  (`test_acc`) is taken at the epoch with the lowest loss **on the test set**, i.e. the test set also
  drives model selection (inherited from the course assignment's `train_model`). This makes absolute
  accuracy optimistic and inflated the depth-20 residual advantage. The final-epoch accuracy is always
  recorded too and involves no selection.
- `cosine` (**v2**): identical except the learning rate follows cosine annealing over the 10 epochs
  (1e-3 down to ~2e-5), and the reported metric is the **final-epoch** test accuracy. No checkpoint is
  selected, so the test set influences no decision. It asks whether the depth-20 advantage persists when
  training is closer to convergence, or is mainly a convergence-speed effect.

Everything else (architecture, parameter counts, seeds, augmentation, weight decay, batch size) is
identical, so with the same seed **epoch 1 is identical under both schedules** (lr = 1e-3 in epoch 1);
the comparison cell checks this. Each schedule writes to its own JSON file, so v1 results are never
overwritten.

This notebook is a standalone research extension of the HW2 assignment
(`AIMultimedia_Practice.ipynb`) and is **not** part of the graded submission. It reuses the
same helper-function conventions (`train`, `evaluate`, `train_model`, etc.) for consistency.

**Structure:**
1. Setup (imports, seed, Colab Drive mount, data loaders, helpers incl. the `schedule` option)
2. `ToggleBlock` / `ToggleResNet` architecture (He et al. 2016 CIFAR-ResNet parameterization,
   `depth = 6n + 2`, option-A zero-padding shortcut so parameter count is identical for
   Residual on vs. off at every depth)
3. The experiment — depth ∈ {8, 20} (n ∈ {1, 3}) × Residual ∈ {on, off}, BN fixed on, 3 seeds; set `SCHEDULE`
4. Analysis of the chosen schedule (final-epoch and legacy best-loss-epoch metrics, learning curves)
5. v1 vs v2 comparison (constant vs cosine) incl. the epoch-1 sanity check
6. Gradient-norm diagnostics at both depths
7. Summary


In [ ]:
# === Setup — run this cell first ===
import os, math, random, json, time, collections
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

BATCH_SIZE = 64
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

# --- Persistence: mount Google Drive on Colab so results/checkpoints/dataset survive
# disconnects and don't need re-downloading on every fresh Colab VM ---
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/BN_Residual_Depth_Ablation/'
    RESULTS_DIR = BASE_DIR
    DATA_DIR = BASE_DIR + 'data'
    os.makedirs(RESULTS_DIR, exist_ok=True)
    print('Running on Colab — results/checkpoints/dataset will be saved to Google Drive:', BASE_DIR)
except ImportError:
    RESULTS_DIR = './'
    DATA_DIR = './data'
    print('Not running on Colab — results will be saved locally to', RESULTS_DIR)

CKPT_DIR = RESULTS_DIR

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

# download=True only actually downloads if DATA_DIR doesn't already contain a valid copy —
# once this has run once with DATA_DIR on Google Drive, later sessions reuse it instead of
# re-downloading from the network every time.
trainset = datasets.CIFAR10(root=DATA_DIR, train=True, download=True, transform=train_transform)
testset  = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=test_transform)
train_iterator = data.DataLoader(trainset, shuffle=True, batch_size=BATCH_SIZE, num_workers=2)
test_iterator  = data.DataLoader(testset,  batch_size=BATCH_SIZE, num_workers=2)
print(f'Train: {len(trainset)} | Test: {len(testset)} | Classes: {trainset.classes}')

def calculate_accuracy(y_pred, y):
    top_pred = y_pred.argmax(1, keepdim=True)
    correct = top_pred.eq(y.view_as(top_pred)).sum()
    return correct.float() / y.shape[0]

def train(model, iterator, optimizer, criterion, device):
    epoch_loss = epoch_acc = 0
    model.train()
    for x, y in tqdm(iterator, desc='Training', leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        y_pred = model(x)
        loss = criterion(y_pred, y)
        acc = calculate_accuracy(y_pred, y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        epoch_acc += acc.item()
    return epoch_loss / len(iterator), epoch_acc / len(iterator)

def evaluate(model, iterator, criterion, device):
    epoch_loss = epoch_acc = 0
    model.eval()
    with torch.no_grad():
        for x, y in tqdm(iterator, desc='Evaluating', leave=False):
            x, y = x.to(device), y.to(device)
            y_pred = model(x)
            loss = criterion(y_pred, y)
            acc = calculate_accuracy(y_pred, y)
            epoch_loss += loss.item()
            epoch_acc += acc.item()
    return epoch_loss / len(iterator), epoch_acc / len(iterator)

def epoch_time(start_time, end_time):
    elapsed = end_time - start_time
    return int(elapsed / 60), int(elapsed % 60)

def train_model(model, train_iterator, test_iterator, device,
                epochs=8, lr=0.001, optimizer_name='adam',
                weight_decay=0.0, model_path='best-cifar10-model.pt', verbose=True,
                schedule='constant'):
    model = model.to(device)
    criterion = nn.CrossEntropyLoss().to(device)
    if optimizer_name.lower() == 'sgd':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    else:
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    if schedule == 'cosine':
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    elif schedule == 'constant':
        scheduler = None
    else:
        raise ValueError(f"unknown schedule: {schedule!r} (use 'constant' or 'cosine')")
    best_valid_loss = float('inf')
    train_losses, train_accuracies, valid_losses, valid_accuracies, lrs = [], [], [], [], []
    for epoch in tqdm(range(epochs)):
        lrs.append(optimizer.param_groups[0]['lr'])
        start_time = time.monotonic()
        train_loss, train_acc = train(model, train_iterator, optimizer, criterion, device)
        valid_loss, valid_acc = evaluate(model, test_iterator, criterion, device)
        if schedule == 'constant':
            # v1 convention: keep the epoch with the lowest loss on the evaluation set
            if valid_loss < best_valid_loss:
                best_valid_loss = valid_loss
                torch.save(model.state_dict(), model_path)
        elif epoch == epochs - 1:
            # cosine: no selection; the model after the final epoch is the reported model
            torch.save(model.state_dict(), model_path)
        if scheduler is not None:
            scheduler.step()
        end_time = time.monotonic()
        epoch_mins, epoch_secs = epoch_time(start_time, end_time)
        if verbose:
            print(f'Epoch: {epoch+1:02} | Epoch Time: {epoch_mins}m {epoch_secs}s')
            print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
            print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')
        train_losses.append(train_loss); train_accuracies.append(train_acc)
        valid_losses.append(valid_loss); valid_accuracies.append(valid_acc)
    return {
        'train_losses': train_losses, 'train_accuracies': train_accuracies,
        'valid_losses': valid_losses, 'valid_accuracies': valid_accuracies,
        'best_model_path': model_path, 'lrs': lrs
    }

def savefig(name):
    plt.savefig(RESULTS_DIR + name, dpi=150, bbox_inches='tight')

def append_result(path, result):
    if os.path.exists(path):
        with open(path, 'r') as f:
            all_results = json.load(f)
    else:
        all_results = []
    all_results.append(result)
    with open(path, 'w') as f:
        json.dump(all_results, f, indent=2)
    return all_results

def load_results(path):
    return json.load(open(path)) if os.path.exists(path) else []

print('Setup complete.')


---
## Architecture: `ToggleBlock` / `ToggleResNet`

Follows He et al. (2016) CIFAR-10 ResNet parameterization exactly: `depth = 6n + 2`
(1 stem conv + 3 stages × n blocks × 2 convs + 1 FC), 3 stages with widths [16, 32, 64].
`use_bn` is kept as a constructor flag (for future work extending back to the full BN×Residual
factorial) but this notebook always calls it with `use_bn=True`.

Downsampling/widening shortcuts use **option A** (zero-padding identity, matching the
original CIFAR experiment) — this adds **zero parameters**, so `use_residual=True` and
`use_residual=False` have **exactly the same parameter count** at every depth.


In [ ]:
class ToggleBlock(nn.Module):
    """Two 3x3 convs, each optionally followed by BatchNorm. If use_residual, adds a
    skip connection (option-A zero-padding identity when stride!=1 or channels change,
    so this never adds parameters)."""
    def __init__(self, in_channels, out_channels, stride, use_bn, use_residual):
        super().__init__()
        self.use_residual = use_residual
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride,
                                padding=1, bias=not use_bn)
        self.bn1 = nn.BatchNorm2d(out_channels) if use_bn else nn.Identity()
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1,
                                padding=1, bias=not use_bn)
        self.bn2 = nn.BatchNorm2d(out_channels) if use_bn else nn.Identity()

        self.shortcut = None
        if use_residual and (stride != 1 or in_channels != out_channels):
            pad = out_channels - in_channels
            self.shortcut = lambda x: F.pad(x[:, :, ::stride, ::stride],
                                             (0, 0, 0, 0, 0, pad), 'constant', 0)
        elif use_residual:
            self.shortcut = nn.Identity()

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.use_residual:
            out = out + self.shortcut(x)
        return F.relu(out)


class ToggleResNet(nn.Module):
    """CIFAR-ResNet-style network, depth = 6n + 2. use_bn / use_residual are global
    switches applied identically to every block."""
    def __init__(self, n, use_bn=True, use_residual=True, num_classes=10):
        super().__init__()
        widths = [16, 32, 64]
        self.conv1 = nn.Conv2d(3, widths[0], kernel_size=3, stride=1, padding=1, bias=not use_bn)
        self.bn1 = nn.BatchNorm2d(widths[0]) if use_bn else nn.Identity()

        self.stage1 = self._make_stage(widths[0], widths[0], n, stride=1, use_bn=use_bn, use_residual=use_residual)
        self.stage2 = self._make_stage(widths[0], widths[1], n, stride=2, use_bn=use_bn, use_residual=use_residual)
        self.stage3 = self._make_stage(widths[1], widths[2], n, stride=2, use_bn=use_bn, use_residual=use_residual)

        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(widths[2], num_classes)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.constant_(m.weight, 1)
            nn.init.constant_(m.bias, 0)

    def _make_stage(self, in_channels, out_channels, n_blocks, stride, use_bn, use_residual):
        layers = [ToggleBlock(in_channels, out_channels, stride, use_bn, use_residual)]
        for _ in range(1, n_blocks):
            layers.append(ToggleBlock(out_channels, out_channels, 1, use_bn, use_residual))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)


def count_params(model):
    return sum(p.numel() for p in model.parameters())

def depth_of(n):
    return 6 * n + 2

# Sanity check: parameter count must be identical across use_residual at both depths we test.
print('Parameter count sanity check (BN fixed on):')
for n in [1, 3]:
    for use_residual in [True, False]:
        m = ToggleResNet(n=n, use_bn=True, use_residual=use_residual)
        print(f'  depth={depth_of(n):3} Residual={use_residual!s:5} -> {count_params(m):,} params')


---
## `run_single`: one training run

Thin wrapper around the existing `train_model()` — sets the seed (controls both weight
initialization and DataLoader shuffle order), builds the model, trains it with the chosen
`schedule`, and returns a result dict.

Each record stores two accuracies: `test_acc` (legacy v1 metric: accuracy at the epoch with the
lowest loss on the test set — this **selects on the test set**) and `final_test_acc` (accuracy after
the last epoch — **no selection**). The `cosine` recipe is analysed with `final_test_acc`.


In [ ]:
def run_single(n, use_bn, use_residual, seed, epochs=10, schedule='constant', verbose=True):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed(seed)

    g = torch.Generator(); g.manual_seed(seed)
    seeded_train_iterator = data.DataLoader(
        trainset, shuffle=True, batch_size=BATCH_SIZE, num_workers=2, generator=g)

    model = ToggleResNet(n=n, use_bn=use_bn, use_residual=use_residual)
    suffix = '' if schedule == 'constant' else f'_{schedule}'   # keep v1 checkpoint names unchanged
    model_path = f'{CKPT_DIR}toggle_n{n}_bn{int(use_bn)}_res{int(use_residual)}_seed{seed}{suffix}.pt'

    stats = train_model(model, seeded_train_iterator, test_iterator, device,
                         epochs=epochs, lr=0.001, optimizer_name='adam',
                         weight_decay=1e-4, model_path=model_path, verbose=False,
                         schedule=schedule)

    best_idx = int(np.argmin(stats['valid_losses']))
    result = {
        'n': n, 'depth': depth_of(n), 'use_bn': use_bn, 'use_residual': use_residual,
        'seed': seed, 'epochs': epochs, 'schedule': schedule, 'num_params': count_params(model),
        'best_epoch': best_idx + 1,
        'test_acc': stats['valid_accuracies'][best_idx],        # legacy: selects on the test set
        'test_loss': stats['valid_losses'][best_idx],
        'final_test_acc': stats['valid_accuracies'][-1],        # no selection
        'final_test_loss': stats['valid_losses'][-1],
        'lrs': stats['lrs'],
        'train_losses': stats['train_losses'], 'train_accuracies': stats['train_accuracies'],
        'valid_losses': stats['valid_losses'], 'valid_accuracies': stats['valid_accuracies'],
    }
    if verbose:
        print(f"[{schedule} n={n} depth={depth_of(n)} BN={use_bn} Res={use_residual} seed={seed}] "
              f"params={result['num_params']:,} final_acc={result['final_test_acc']*100:.2f}% "
              f"(best-loss-epoch acc={result['test_acc']*100:.2f}% @ epoch {result['best_epoch']}/{epochs})")
    return result


---
## The experiment: Residual × Depth, BN fixed ON

**12 runs per schedule**: depth ∈ {8, 20} (n ∈ {1, 3}) × Residual ∈ {on, off} × 3 seeds, 10 epochs
each. BN is fixed to `True` for every run — the BN on/off axis is out of scope for this design.

**Set `SCHEDULE` below:**

- `'constant'` — v1. Its results already exist (`s8_residual_depth_results.json`), so re-running with this
  setting skips everything and just re-loads them.
- `'cosine'` — v2 (new). Results go to `s8_residual_depth_results_cosine.json`. **This is the run to do now.**

Results are appended to JSON after every single run, and already-completed
(n, use_residual, seed) combinations are skipped on re-run, so a Colab disconnect never
forces a full restart. To add more seeds, extend `EXP_SEEDS` (e.g. `2029, 2030`); do the same for
the other schedule if you want the two recipes to keep matched seed counts.


In [ ]:
SCHEDULE = 'cosine'          # 'constant' = v1 (already done), 'cosine' = v2
EXP_PATH = RESULTS_DIR + ('s8_residual_depth_results.json' if SCHEDULE == 'constant'
                          else f's8_residual_depth_results_{SCHEDULE}.json')
EXP_NS = [1, 3]              # depth 8 and depth 20
EXP_SEEDS = [2026, 2027, 2028]
EXP_EPOCHS = 10
EXP_USE_BN = True            # fixed for this design

if SCHEDULE == 'cosine':
    lr_preview = [0.5 * 1e-3 * (1 + math.cos(math.pi * k / EXP_EPOCHS)) for k in range(EXP_EPOCHS)]
    print('cosine learning rate per epoch:', [f'{v:.1e}' for v in lr_preview])

existing = load_results(EXP_PATH)
done = {(r['n'], r['use_residual'], r['seed']) for r in existing}

for n in EXP_NS:
    for use_residual in [True, False]:
        for seed in EXP_SEEDS:
            if (n, use_residual, seed) in done:
                print(f'skip (already done): {SCHEDULE} depth={depth_of(n)} Res={use_residual} seed={seed}')
                continue
            result = run_single(n, EXP_USE_BN, use_residual, seed, epochs=EXP_EPOCHS, schedule=SCHEDULE)
            append_result(EXP_PATH, result)

print(f'Experiment ({SCHEDULE}): {len(load_results(EXP_PATH))} / {len(EXP_NS) * 2 * len(EXP_SEEDS)} runs complete.')


In [ ]:
def final_acc(r):
    return r['valid_accuracies'][-1]     # accuracy after the last epoch: no checkpoint selection
def best_acc(r):
    return r['test_acc']                 # accuracy at the lowest-test-loss epoch: selects on the test set

results = load_results(EXP_PATH)
summ = {}
for n in EXP_NS:
    for use_res in (True, False):
        rr = sorted([r for r in results if r['n'] == n and r['use_residual'] == use_res], key=lambda r: r['seed'])
        summ[(n, use_res)] = {'final': np.array([final_acc(r) for r in rr]) * 100,
                              'best': np.array([best_acc(r) for r in rr]) * 100,
                              'runs': rr}

primary = 'final' if SCHEDULE == 'cosine' else 'best'
print(f'Schedule = {SCHEDULE}; {len(results)} runs loaded from {EXP_PATH}')
print('Primary metric:', 'final-epoch accuracy (no selection)' if primary == 'final'
      else 'accuracy at lowest-test-loss epoch (v1 convention; final-epoch shown too)')
print()

deltas = {}
for metric, title in (('final', 'FINAL-EPOCH test accuracy (no selection)'),
                      ('best', 'BEST-TEST-LOSS-EPOCH test accuracy (selects on the test set)')):
    print(f'=== {title}: mean ± SD (n-1) ===')
    for n in EXP_NS:
        on, off = summ[(n, True)][metric], summ[(n, False)][metric]
        if len(on) < 2 or len(off) < 2:
            print(f'depth={depth_of(n):3} incomplete (ON n={len(on)}, OFF n={len(off)})')
            continue
        deltas[(metric, n)] = on.mean() - off.mean()
        print(f'depth={depth_of(n):3} ON {on.mean():6.2f} ± {on.std(ddof=1):4.2f} {np.round(on, 2)} | '
              f'OFF {off.mean():6.2f} ± {off.std(ddof=1):4.2f} {np.round(off, 2)} | '
              f'Δ(ON-OFF) {deltas[(metric, n)]:+.2f}pp')
    if all((metric, n) in deltas for n in EXP_NS) and len(EXP_NS) == 2:
        print(f'  difference-in-differences (depth {depth_of(EXP_NS[1])} Δ − depth {depth_of(EXP_NS[0])} Δ) = '
              f'{deltas[(metric, EXP_NS[1])] - deltas[(metric, EXP_NS[0])]:+.2f}pp')
    print()

for n in EXP_NS:
    a = [r['valid_accuracies'] for r in summ[(n, True)]['runs']]
    b = [r['valid_accuracies'] for r in summ[(n, False)]['runs']]
    if a and b:
        print(f'depth {depth_of(n)}: per-epoch mean validation-accuracy gap (ON − OFF), pp:',
              np.round((np.array(a).mean(0) - np.array(b).mean(0)) * 100, 1))
print()

# ---- interaction plot for the primary metric ----
fig, ax = plt.subplots(figsize=(5.2, 4.0))
for use_res, color, lab, dx in ((True, '#1f77b4', 'Residual ON', -0.25), (False, '#d95f02', 'Residual OFF', 0.25)):
    xs, means, sds = [], [], []
    for n in EXP_NS:
        a = summ[(n, use_res)][primary]
        if len(a) == 0:
            continue
        ax.scatter([depth_of(n) + dx] * len(a), a, color=color, alpha=0.45, s=22, zorder=3)
        xs.append(depth_of(n) + dx); means.append(a.mean()); sds.append(a.std(ddof=1) if len(a) > 1 else 0.0)
    ax.errorbar(xs, means, yerr=sds, color=color, marker='o', capsize=4, lw=1.6, label=lab, zorder=4)
ax.set_xticks([depth_of(n) for n in EXP_NS]); ax.set_xlabel('Depth (6n+2)'); ax.set_ylabel('Test accuracy (%)')
ax.set_title(f'{SCHEDULE}: ' + ('final-epoch' if primary == 'final' else 'best-test-loss-epoch') +
             ' accuracy\nmean ± SD (n-1); dots = seeds', fontsize=10)
ax.grid(alpha=0.25); ax.legend(frameon=False); fig.tight_layout()
savefig(f'interaction_{SCHEDULE}.png'); plt.show()

# ---- learning curves: mean over seeds, band = min-max across seeds ----
fig, axes = plt.subplots(3, len(EXP_NS), figsize=(4.6 * len(EXP_NS), 8.4), sharex=True, squeeze=False)
for j, n in enumerate(EXP_NS):
    lr_curve = None
    for use_res, color, lab in ((True, '#1f77b4', 'Residual ON'), (False, '#d95f02', 'Residual OFF')):
        rr = summ[(n, use_res)]['runs']
        if not rr:
            continue
        va = np.array([r['valid_accuracies'] for r in rr]) * 100
        tl = np.array([r['train_losses'] for r in rr])
        ep = np.arange(1, va.shape[1] + 1)
        axes[0, j].plot(ep, va.mean(0), color=color, label=lab)
        axes[0, j].fill_between(ep, va.min(0), va.max(0), color=color, alpha=0.18)
        axes[1, j].plot(ep, tl.mean(0), color=color)
        axes[1, j].fill_between(ep, tl.min(0), tl.max(0), color=color, alpha=0.18)
        if lr_curve is None and 'lrs' in rr[0]:
            lr_curve = (ep, rr[0]['lrs'])
    if lr_curve is not None:
        axes[2, j].plot(*lr_curve, color='gray', marker='o', ms=3)
    axes[0, j].set_title(f'depth {depth_of(n)}: validation accuracy (%)', fontsize=10)
    axes[1, j].set_title(f'depth {depth_of(n)}: training loss', fontsize=10)
    axes[2, j].set_title('learning rate', fontsize=10); axes[2, j].set_xlabel('Epoch')
    for i in range(3):
        axes[i, j].grid(alpha=0.25)
axes[0, 0].legend(frameon=False)
fig.suptitle(f'schedule = {SCHEDULE}', fontsize=10)
fig.tight_layout(rect=(0, 0, 1, 0.97)); savefig(f'learning_curves_{SCHEDULE}.png'); plt.show()


---
## v1 (constant LR) vs. v2 (cosine LR)

Needs both JSON files (run the experiment cell once with each `SCHEDULE`). Compares the
**final-epoch** accuracy (no selection) under the two recipes, and checks that epoch 1 is
identical across recipes for the same (depth, residual, seed).


In [ ]:
V1_PATH = RESULTS_DIR + 's8_residual_depth_results.json'
V2_PATH = RESULTS_DIR + 's8_residual_depth_results_cosine.json'
v1, v2 = load_results(V1_PATH), load_results(V2_PATH)

if not v1 or not v2:
    print('Need both files (constant + cosine). Run the experiment cell with each SCHEDULE first.')
else:
    def cells_of(res):
        return {(n, ur): np.array([r['valid_accuracies'][-1] for r in res
                                   if r['n'] == n and r['use_residual'] == ur]) * 100
                for n in EXP_NS for ur in (True, False)}

    print('Final-epoch test accuracy (%), mean ± SD (n-1)')
    print(f"{'recipe':9s} {'depth':>5s} {'Residual ON':>16s} {'Residual OFF':>16s} {'Δ (ON-OFF)':>11s}")
    delta_by = {}
    for label, res in (('constant', v1), ('cosine', v2)):
        c = cells_of(res)
        delta_by[label] = []
        for n in EXP_NS:
            on, off = c[(n, True)], c[(n, False)]
            d = on.mean() - off.mean()
            delta_by[label].append(d)
            print(f"{label:9s} {depth_of(n):5d} {on.mean():8.2f} ± {on.std(ddof=1):4.2f} "
                  f"{off.mean():9.2f} ± {off.std(ddof=1):4.2f} {d:+11.2f}")
        if len(EXP_NS) == 2:
            print(f"{label:9s} difference-in-differences = {delta_by[label][1] - delta_by[label][0]:+.2f}pp")
    print()

    plt.figure(figsize=(5.2, 4.0))
    for label, marker in (('constant', 'o'), ('cosine', 's')):
        plt.plot([depth_of(n) for n in EXP_NS], delta_by[label], marker=marker, label=label)
    plt.axhline(0, color='gray', ls='--', lw=1)
    plt.xticks([depth_of(n) for n in EXP_NS]); plt.xlabel('Depth (6n+2)')
    plt.ylabel('Δ final-epoch accuracy, Residual ON − OFF (pp)')
    plt.title('Effect of the skip connection by training recipe', fontsize=10)
    plt.grid(alpha=0.25); plt.legend(frameon=False); plt.tight_layout()
    savefig('delta_by_schedule.png'); plt.show()

    key = lambda r: (r['n'], r['use_residual'], r['seed'])
    m1 = {key(r): r for r in v1}
    diffs = []
    for r in v2:
        q = m1.get(key(r))
        if q is not None:
            diffs.append(max(abs(r['train_losses'][0] - q['train_losses'][0]),
                             abs(r['valid_losses'][0] - q['valid_losses'][0])))
    if diffs:
        print(f'Epoch-1 check: {len(diffs)} matched runs, max |Δ| in epoch-1 train/valid loss = {max(diffs):.2e}')
        print('Expected ~0 on the same GPU type: epoch 1 uses lr = 1e-3 under both schedules with the same seed.')
        print('A tiny nonzero value can come from a different GPU/driver; a large one means the recipes')
        print('differ in more than the learning-rate schedule.')
    else:
        print('No matching (depth, residual, seed) runs between the two files for the epoch-1 check.')


---
## Gradient-norm diagnostics

At both depths (n=1, n=3), BN fixed on, measure the per-conv-layer gradient L2 norm at
initialization, averaged over the first `num_batches` training batches, for Residual on vs.
off. This is a lightweight stand-in for the fuller Santurkar-style loss-landscape
diagnostics (stretch goal, not implemented here) — it answers "does the plain network's
early-layer gradient collapse relative to the residual network's, and does that gap change
between depth 8 and depth 20?"


In [ ]:
def measure_gradient_norms(n, use_bn, use_residual, seed, num_batches=50):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = ToggleResNet(n=n, use_bn=use_bn, use_residual=use_residual).to(device)
    criterion = nn.CrossEntropyLoss().to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    model.train()

    conv_layers = [(name, p) for name, p in model.named_parameters()
                   if p.requires_grad and p.dim() == 4]  # conv weight tensors only
    norm_history = {name: [] for name, _ in conv_layers}

    it = iter(train_iterator)
    for _ in range(num_batches):
        try:
            x, y = next(it)
        except StopIteration:
            it = iter(train_iterator)
            x, y = next(it)
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        for name, p in conv_layers:
            norm_history[name].append(p.grad.norm().item())
        optimizer.step()

    return {name: float(np.mean(v)) for name, v in norm_history.items()}


GRAD_SEED = 2026

for n in EXP_NS:
    plt.figure(figsize=(7, 4))
    for use_residual in [True, False]:
        norms = measure_gradient_norms(n, EXP_USE_BN, use_residual, GRAD_SEED)
        layer_names = list(norms.keys())
        plt.plot(range(len(layer_names)), list(norms.values()), marker='o',
                  label=f'Residual={use_residual}')
    plt.yscale('log')
    plt.xlabel('Conv layer index (input -> output)')
    plt.ylabel('Mean gradient L2 norm (log scale)')
    plt.title(f'Gradient norm by layer, depth={depth_of(n)} (BN=on)')
    plt.legend()
    plt.tight_layout()
    savefig(f'gradient_norms_depth{depth_of(n)}.png')
    plt.show()


---
## Summary

Prints the results of the schedule selected above. This does **not** write to `report.txt`/`report.md`
automatically. After the `cosine` run finishes, download the results folder from Drive and share
`s8_residual_depth_results_cosine.json` so the full statistics (Welch tests, interaction ANOVA,
confidence intervals) can be computed and the report updated. Remember to keep noting that the BN on/off
axis (H1a/H1b) was out of scope for this design and is left as future work.


In [ ]:
print(f'=== Schedule: {SCHEDULE} (BN fixed ON, {len(EXP_SEEDS)} seeds, {EXP_EPOCHS} epochs) ===')
for n in EXP_NS:
    for use_res in (True, False):
        f, b = summ[(n, use_res)]['final'], summ[(n, use_res)]['best']
        if len(f) > 1:
            print(f'  depth={depth_of(n):3} Residual={"ON " if use_res else "OFF"}: '
                  f'final-epoch {f.mean():.2f} ± {f.std(ddof=1):.2f}%   '
                  f'best-test-loss-epoch {b.mean():.2f} ± {b.std(ddof=1):.2f}%')

print()
print('Scope reminder: BN on/off (H1a/H1b) was not tested in this design — future work.')
